In [1]:
import pandas as pd
import os
from pathlib import Path
import pprint
import csv

In [2]:
def read_metadata(file_path):
    with open(file_path, 'r') as f:
        # Create a generator that filters out lines starting with #
        filtered_file = (line for line in f if line.startswith('#'))

        # Pass that generator to the DictReader
        reader = csv.DictReader(filtered_file, delimiter='\t')

        data = list(reader)
        # Display in a readable format
        pprint.pprint(data)

# Transcriptional


In [3]:
def consolidate_gdc_data(sample_sheet_path, data_dir, output_path, data_type_filter):
    """
    Consolidates individual GDC files into a single Parquet file.

    Args:
        sample_sheet_path (str): Absolute path to the GDC sample sheet.
        data_dir (str): Absolute path to the directory containing the 430 folders/files.
        output_path (str): Where to save the resulting Parquet file.
        data_type_filter (str): 'Gene Expression Quantification' or 'miRNA Expression Quantification'
    """
    # 1. Load the Sample Sheet
    sheet = pd.read_csv(sample_sheet_path, sep='\t')

    # Filter for the specific data type (RNA-seq or miRNA-seq)
    tumor_samples = sheet[sheet['Tissue Type'].str.contains('Tumor', case=False)].copy()
    filtered_sheet = tumor_samples[tumor_samples['Data Type'] == data_type_filter].copy()

    all_data = []

    print(f"Starting processing for {data_type_filter}...")

    for index, row in filtered_sheet.iterrows():
        file_id = row['File ID']
        file_name = row['File Name']

        file_path = os.path.join(data_dir, file_id, file_name)

        if not os.path.exists(file_path):
            print(f"Warning: File not found: {file_path}")
            continue

        # 2. Read the specific file
        if "miRNA" in data_type_filter:
            df = pd.read_csv(file_path, sep='\t')
            id_col = df.columns[0]
            val_col = 'reads_per_million_miRNA_mapped'
        else:
            # RNA-seq files: skip metadata lines
            df = pd.read_csv(file_path, sep='\t', comment='#')
            id_col = 'gene_id'
            val_col = 'tpm_unstranded'

        # The choice of "value" can be further investigated by looking at there distribution in crosspondence to the used model

        # Pivot the data to make genes the columns
        temp_df = df[[id_col, val_col]].set_index(id_col).T

        # 3. Inject Metadata
        temp_df.insert(0, 'patient_id', row['Case ID'])

        all_data.append(temp_df)

    # 4. Concatenate and Export
    final_df = pd.concat(all_data, axis=0).reset_index(drop=True)
    final_df.to_parquet(output_path, engine='pyarrow')
    print(f"Success! Saved {len(final_df)} samples to {output_path}")



In [4]:
# --- EXECUTION ---
SAMPLE_SHEET = "/media/maro/Mom0-0/Datasets/TCGA/TCGA-COAD-transcriptional/gdc_sample_sheet.2026-05-14.tsv"
RNA_DIR = "/media/maro/Mom0-0/Datasets/TCGA/TCGA-COAD-transcriptional"
MIRNA_DIR = "/media/maro/Mom0-0/Datasets/TCGA/TCGA-COAD-transcriptional"



In [5]:
# Process RNA-Seq
consolidate_gdc_data(
    sample_sheet_path=SAMPLE_SHEET,
    data_dir=RNA_DIR,
    output_path="../../../data/raw/coad_rna_consolidated.parquet",
    data_type_filter="Gene Expression Quantification"
)


Starting processing for Gene Expression Quantification...
Success! Saved 394 samples to ../../../data/raw/coad_rna_consolidated.parquet


In [5]:
# Process miRNA-Seq
consolidate_gdc_data(
    sample_sheet_path=SAMPLE_SHEET,
    data_dir=MIRNA_DIR,
    output_path="../../../data/raw/coad_mirna_consolidated.parquet",
    data_type_filter="miRNA Expression Quantification"
)

Starting processing for miRNA Expression Quantification...
Success! Saved 446 samples to ../../../data/processed/stad_mirna_consolidated.parquet


In [22]:
read_metadata(SAMPLE_SHEET)

[]


# Clinical Patient

In [3]:
clinical_patient_path = "/media/maro/Mom0-0/Datasets/TCGA/TMB Prediction/downloads_valid/coad_tcga_gdc/data_clinical_patient.txt"

In [4]:
clinical_patient_df = pd.read_csv(clinical_patient_path, sep='\t', comment='#')
clinical_patient_df.columns

Index(['PATIENT_ID', 'AGE', 'AJCC_STAGING_EDITION', 'BIOPSY_SITE',
       'DAYS_LAST_FOLLOWUP', 'DAYS_TO_BIRTH', 'DAYS_TO_DEATH', 'DISEASE_TYPE',
       'ETHNICITY', 'ICD_10', 'MORPHOLOGY', 'OTHER_PATIENT_ID', 'PATH_M_STAGE',
       'PATH_N_STAGE', 'PATH_STAGE', 'PATH_T_STAGE', 'PRIMARY_DIAGNOSIS',
       'PRIMARY_SITE_PATIENT', 'PRIOR_MALIGNANCY', 'PRIOR_TREATMENT', 'RACE',
       'SEX', 'VITAL_STATUS', 'YEAR_OF_DIAGNOSIS', 'OS_STATUS', 'OS_MONTHS',
       'DFS_STATUS', 'DFS_MONTHS', 'PROJECT_ID', 'PROJECT_NAME',
       'PROJECT_STATE'],
      dtype='str')

In [5]:
read_metadata(clinical_patient_path)

[{'#Patient Identifier': '#Identifier to uniquely specify a patient.',
  'AJCC Pathologic M-Stage': 'Code to represent the defined absence or '
                             'presence of distant spread or metastases (M) to '
                             'locations via vascular channels or lymphatics '
                             'beyond the regional lymph nodes, using criteria '
                             'established by the American Joint Committee on '
                             'Cancer (AJCC).',
  'AJCC Pathologic N-Stage': 'The codes that represent the stage of cancer '
                             'based on the nodes present (N stage) according '
                             'to criteria based on multiple editions of the '
                             "AJCC's Cancer Staging Manual.",
  'AJCC Pathologic Stage': 'The extent of a cancer, especially whether the '
                           'disease has spread from the original site to other '
                           'parts of t

In [7]:
clinical_patient_df.drop(columns=['OTHER_PATIENT_ID'], inplace=True)

In [8]:
clinical_patient_df.to_parquet("../../../data/raw/coad_clinical_patient.parquet", engine='pyarrow')

# TMB Target

In [27]:
clinical_sample_path = "/media/maro/Mom0-0/Datasets/TCGA/TMB Prediction/downloads_valid/coad_tcga_gdc/data_clinical_sample.txt"


In [28]:
clinical_sample_df = pd.read_csv(clinical_sample_path, sep='\t', comment='#')
clinical_sample_df.columns

Index(['PATIENT_ID', 'SAMPLE_ID', 'ONCOTREE_CODE', 'CANCER_TYPE',
       'CANCER_TYPE_DETAILED', 'TUMOR_TYPE', 'GRADE',
       'TISSUE_PROSPECTIVE_COLLECTION_INDICATOR',
       'TISSUE_RETROSPECTIVE_COLLECTION_INDICATOR', 'TISSUE_SOURCE_SITE_CODE',
       'TUMOR_TISSUE_SITE', 'ANEUPLOIDY_SCORE', 'SAMPLE_TYPE',
       'MSI_SCORE_MANTIS', 'MSI_SENSOR_SCORE', 'SOMATIC_STATUS',
       'TMB_NONSYNONYMOUS', 'TISSUE_SOURCE_SITE', 'TBL_SCORE'],
      dtype='str')

In [29]:
clinical_sample_df[['PATIENT_ID','TMB_NONSYNONYMOUS']].to_parquet("../../../data/raw/coad_tmb.parquet", engine='pyarrow')
